In [ ]:
import os, sys, shutil, subprocess, re, glob 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make GRASS Python packages importable.
sys.path.append(
    subprocess.check_output(["grass85", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj

In [ ]:
GISDBASE = "/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Notebooks"

In [ ]:
session=gj.init(GISDBASE, location="DEM_generation", mapset="for_codem")

In [ ]:
#!g.list type=

In [ ]:
!g.region raster=hull_mask res=1

In [ ]:
gs.run_command("r.mask", raster="hull_mask", overwrite=True)

In [ ]:
gs.run_command("r.watershed", elevation="lidar_2020_DTM_ref", accumulation="accumulation", drainage="drainage", basin="basins_90", threshold=90000, overwrite=True)

In [ ]:
basin_rast = "basins_90"          # <-- your r.watershed basin raster
basin_vect = f"{basin_rast}_v"         # output vector

# raster -> vector polygons
gs.run_command(
    "r.to.vect",
    input=basin_rast,
    output=basin_vect,
    type="area",
    overwrite=True
)
# add a label field and copy cat into it (basin id)
try:
    gs.run_command("v.db.addcolumn", map=basin_vect, columns="basin_id integer")
except gs.CalledModuleError:
    pass

gs.run_command("v.db.update", map=basin_vect, column="basin_id", query_column="cat")

print("Created:", basin_vect)

In [ ]:
basin_vect = "basins_90_v"
basin_cent = "basins_90_cent"

gs.run_command("v.centroids", input=basin_vect, output=basin_cent, overwrite=True)

m = gj.Map()
m.d_vect(map=basin_vect, color="black", width=1)
m.d_vect(map=basin_cent, display="cat", color="red", size=12)
m.show()


In [ ]:
basin_vect = "basins_90_v"
basin_id = 34
out = f"{basin_vect}_cat{basin_id}"
gs.run_command("v.extract", input=basin_vect, output=out, cats=basin_id, overwrite=True)
print("Created:", out)

In [ ]:
gs.run_command("r.mask", flags="r")

In [ ]:
!g.region vector=basins_90_v_cat34

In [ ]:
gs.run_command("r.mask", vector="basins_90_v_cat34", overwrite=True)

In [ ]:
m = gj.Map(use_region=True)
#m.d_rast(map="Ortho_aerial_2024_rgb@DTM_DSM")
#m.d_rast(map="accumulation")
#m.d_rast(map="drainage")
m.d_vect(map="basins_90_v_cat34", color="green", width=2, fill_color="none")
#m.save("basin34_ortho.png")
m.show()

In [ ]:
# !g.list type=raster

In [ ]:
m = gj.Map(use_region=True)
m.d_rast(map="lidar_2020_DTM@DTM_DSM")
m.d_rast(map="lidar_2020_DTM_relief@DTM_DSM")
m.d_vect(map="basins_90_v_cat34", color="green", width=2, fill_color="none")
m.save("basin34_hillshade.jpg")
m.show()

In [ ]:
gs.run_command('r.slope.aspect', elevation="lidar_2020_DTM@DTM_DSM", dx="dx_20", dy="dy_20", overwrite=True)

In [ ]:
gs.run_command('r.sim.water', elevation="lidar_2020_DTM@DTM_DSM", dx="dx_20", dy="dy_20", depth="depth_mask_20", discharge="discharge_20" output_step=1, flags="t", niterations=30)
# Create a time series
gs.run_command("t.create",
               output="depth_mask_20",
               temporaltype="relative",
               title="Overland flow depth",
               description="Overland flow depth")


In [ ]:
# Register the time series
maps = gs.list_strings(type="raster", pattern="depth_mask_20*")
gs.run_command("t.register", input="depth_mask_20", maps=maps)

In [ ]:
#!g.list type=raster

In [ ]:
# flow_map = gj.TimeSeriesMap()
# flow_map.add_raster_series("depth_mask_20")
# flow_map.save("depth_mask_20.gif")
# flow_map.show()

In [ ]:
gs.run_command('r.slope.aspect', elevation="lidar_2024_DTM_corr", dx="dx_24", dy="dy_24", overwrite=True)

In [ ]:
gs.run_command('r.sim.water', elevation="lidar_2024_DTM_corr", dx="dx_24", dy="dy_24", depth="depth_mask_24", output_step=1, flags="t", niterations=30)
# Create a time series
gs.run_command("t.create",
               output="depth_mask_24",
               temporaltype="relative",
               title="Overland flow depth 2024",
               description="Overland flow depth 2024")


In [ ]:
# Register the time series
maps = gs.list_strings(type="raster", pattern="depth_mask_24*")
gs.run_command("t.register", input="depth_mask_24", maps=maps)

In [ ]:
flow_map1 = gj.TimeSeriesMap()
flow_map1.add_raster_series("depth_mask_24")
#flow_map1.add_vect(map="basins_90_v_cat34")
flow_map1.save("depth_mask_24.gif")
flow_map1.show()

In [ ]:
gs.run_command("r.watershed", elevation="lidar_2024_DTM_corr", accumulation="accumulation_2024",  threshold=90000, overwrite=True)

In [ ]:
m = gj.Map()
m.d_vect(map="basins_90_v_cat34", color="red", width=1)
m.d_rast(map="accumulation_2024")
#m.d_legend(raster="accumulation", title="Flow Accum",font="Fira Sans Condensed Light",fontsize=14, flags="-d", title_fontsize=12)
#m.d_barscale(flags="n")
m.save("accumulation_2024.jpg")
m.show()

In [ ]:
gs.run_command("r.mask", flags="r")

### Compute DOD (Erosion and Deposition)

#### DOD between Lidar 2020 and lidar 2024

In [ ]:
#! g.list type=raster

In [ ]:
gs.mapcalc("Change_in_DTM_lidar_may = lidar_2024_regis-lidar_2020_regis", overwrite=True)

In [ ]:
gs.run_command("r.relief", input="Change_in_DTM_lidar_may", output="Change_in_DTM_may_relief",overwrite=True)

In [ ]:
gs.run_command(
    "r.colors",
    map="Change_in_DTM_lidar_may",
    rules="color_corrected"
)

In [ ]:
#!g.region grow=200

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="Change_in_DTM_lidar_may", shade="Change_in_DTM_may_relief", brighten=50,)
#m.d_rast(map="Change_in_DTM_lidar_may")
m.d_vect(map="basins_90_v_cat34", color="green", width=2, fill_color="none")
m.d_legend(raster="Change_in_DTM_lidar_may", title="Elevation Change",font="Fira Sans Condensed Light",fontsize=14, title_fontsize=12)
m.d_barscale(flags="n")
m.save("change_dtm_june_30_b50.jpg")
m.show()

#### DOD between lidar DTM 2020 and sfm DTM 2024

In [ ]:
gs.mapcalc("Change_in_DTM_lidar_cap = cap_sfm_regis-lidar_2020_regis", overwrite=True)

In [ ]:
gs.run_command("r.relief", input="Change_in_DTM_lidar_cap", output="Change_in_DTM_lidar_cap_relief",overwrite=True)

In [ ]:
gs.run_command(
    "r.colors",
    map="Change_in_DTM_lidar_cap",
    rules="color_corrected",
)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="Change_in_DTM_lidar_cap", shade="Change_in_DTM_lidar_cap_relief", brighten=40)
m.d_vect(map="basins_90_v_cat34", color="green", width=2, fill_color="none")
m.d_legend(raster="Change_in_DTM_lidar_cap", title="Elevation Change",font="Fira Sans Condensed Light",fontsize=14, range=(-17.09, 36), title_fontsize=12)
m.d_barscale(flags="n")
m.save("change_dtm_cap_may.jpg")
m.show()

### Overland Flow Maps

#### Overland flow 2020

In [ ]:
gs.run_command(
    "r.slope.aspect",
    elevation="lidar_2020_regis",
    dx="dx_2020",
    dy="dy_2020",
    overwrite=True
)

In [ ]:
gs.run_command('r.sim.water', elevation="lidar_2020_regis", dx="dx_2020", dy="dy_2020", depth="depth_map_2020", discharge="discharge_2020_regis", output_step=4, niterations=48, man_value=0.4, rain_value=50, overwrite=True)

In [ ]:
#gs.run_command("r.relief", input="depth_map", output="depth_map_relief",overwrite=True)
gs.run_command("r.relief", input="discharge_2020_regis", output="discharge_2020_regis_relief",overwrite=True)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="discharge_2020_regis", shade="discharge_2020_regis_relief", brighten=80)
#m.d_rast(map="depth_map")
m.d_vect(map="basins_90_v_cat34", color="red", width=1, fill_color="none")
m.d_legend(raster="discharge_2020_regis", title="Discharge",font="Fira Sans Condensed Light",fontsize=14, flags="-d", title_fontsize=12)
m.d_barscale(flags="n")
m.save("discharge_2020_lidar_may.png")
m.show()

####  Overland flow 2024 lidar

In [ ]:
gs.run_command(
    "r.slope.aspect",
    elevation="lidar_2024_regis",
    dx="dx_24_lidar",
    dy="dy_24_lidar",
    overwrite=True
)

In [ ]:
gs.run_command('r.sim.water', elevation="lidar_2024_regis", dx="dx_24_lidar", dy="dy_24_lidar", depth="depth_may_2024_lidar", discharge="discharge_may_2024_lidar", output_step=4, niterations=48, man_value=0.4, rain_value=50, overwrite=True)

In [ ]:
gs.run_command("r.relief", input="discharge_may_2024_lidar", output="discharge_may_2024_lidar_relief",overwrite=True)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="discharge_may_2024_lidar", shade="discharge_may_2024_lidar_relief", brighten=60)
#m.d_rast(map="depth_map_2024_lidar")
m.d_vect(map="basins_90_v_cat34", color="red", width=1, fill_color="none")
m.d_legend(raster="discharge_may_2024_lidar", title="Discharge",font="Fira Sans Condensed Light",fontsize=14, flags="-d", title_fontsize=12)
m.d_barscale(flags="n")
m.save("discharge_2024_lidar_may.png")
m.show()

####  Overland flow 2024 sfm

In [ ]:
gs.run_command(
    "r.slope.aspect",
    elevation="cap_sfm_regis",
    dx="dx_24_cap",
    dy="dy_24_cap",
    overwrite=True
)

In [ ]:
gs.run_command('r.sim.water', elevation="cap_sfm_regis", dx="dx_24_cap", dy="dy_24_cap", depth="depth_2024_cap", discharge="discharge_2024_cap", output_step=4, niterations=48, man_value=0.4, rain_value=50, overwrite=True)

In [ ]:
gs.run_command("r.relief", input="discharge_2024_cap", output="discharge_2024_cap_relief",overwrite=True)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="discharge_2024_cap", shade="discharge_2024_cap_relief", brighten=60)
#m.d_rast(map="depth_map_2024_sfm")
m.d_vect(map="basins_90_v_cat34", color="red", width=1, fill_color="none")
m.d_legend(raster="discharge_2024_cap", title="Discharge",font="Fira Sans Condensed Light",fontsize=14, flags="-d", title_fontsize=12)
m.d_barscale(flags="n")
m.save("discharge_cap.png")
m.show()

In [ ]:
gs.run_command("r.mask", flags="r")

### lake lure DOD

In [ ]:
#!g.list type=rast

In [ ]:
!g.region raster="boundary" res=1

#### Lidar 2017 and lidar 2024 DoD

In [ ]:
gs.mapcalc("Change_in_elevation_lidar_may_lure = lidar_2024_regis-dtm_2017", overwrite=True)

In [ ]:
gs.run_command("r.relief", input="Change_in_elevation_lidar_may_lure", output="Change_in_elevation_lidar_may_lure_relief",overwrite=True)

In [ ]:
gs.run_command(
    "r.colors",
    map="Change_in_elevation_lidar_may_lure",
    color="differences",
)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="Change_in_elevation_lidar_may_lure", shade="Change_in_elevation_lidar_may_lure_relief", brighten=40)
m.d_legend(raster="Change_in_elevation_lidar_may_lure", flags="-d", title_fontsize=10, at="6,11,97,60")
m.d_barscale(at=(1,12), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.save("change_lidar_may_lure.jpg")
m.show()

#### Lidar 2017 and cap 2024 DoD

In [ ]:
gs.mapcalc("Change_in_elevation_cap_may_lure= cap_sfm_regis-dtm_2017", overwrite=True)

In [ ]:
gs.run_command("r.relief", input="Change_in_elevation_cap_may_lure", output="Change_in_elevation_cap_may_lure_relief",overwrite=True)

In [ ]:
gs.run_command(
    "r.colors",
    map="Change_in_elevation_cap_may_lure",
    color="differences",
)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="Change_in_elevation_cap_may_lure", shade="Change_in_elevation_cap_may_lure_relief", brighten=40)
m.d_legend(raster="Change_in_elevation_cap_may_lure", flags="-d", title_fontsize=10, at="6,11,97,60")
m.d_barscale(at=(1,12), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.save("change_cap_may_lure.jpg")
m.show()